# OSCP Evaluation


### Cell Overview
| Cell | Purpose |
|------|----------|
| 1 | Imports |
| 2 | Config dict |
| 3 | Load classifier |
| 4 | Load SD 1.5 + ControlNet + LCM scheduler + OSCP LoRA |
| 5 | Helper functions (`seed_everything`, `lcm_lora_denoise`, `generate_x_adv`) |
| 6 | Test dataset + DataLoader |
| 7 | Evaluation loop |
| 8 | Print metrics |
| 9 | Save CSV |

## Cell 1 — Imports

In [ ]:
import os
import time
import random
import numpy as np
import torch
import torch.nn.functional as F
import torchvision
import torchvision.transforms.functional as TF
import cv2
import pandas as pd
from tqdm.auto import tqdm
from PIL import Image
from torch.utils.data import DataLoader

# Diffusion pipeline components
from diffusers import (
    LCMScheduler,
    StableDiffusionControlNetImg2ImgPipeline,
    ControlNetModel,
)

# Adversarial attacks
from advertorch.attacks import LinfPGDAttack

try:
    from autoattack import AutoAttack
    _AUTOATTACK_AVAILABLE = True
except ImportError:
    _AUTOATTACK_AVAILABLE = False
    print("[warn] autoattack not installed : AutoAttack will not be available.")
    print("Recommend installing via: pip install git+https://github.com/fra31/auto-attack")

# ---------- Local modules ----------
from archs import get_archs
from dataset import get_dataset
from attack_tools import gen_pgd_confs

# Image I/O helpers from utils.py
from utils import si, mp

print("Imports OK")

Imports OK


## Cell 2 — Config

In [ ]:
config = dict(
    # model/checkpoint paths
    # SD 1.5 base
    pretrained_model  = "stable-diffusion-v1-5/stable-diffusion-v1-5",
    # Canny ControlNet from HuggingFace Hub (or local cache)
    controlnet_model  = "lllyasviel/sd-controlnet-canny",
    # Path to the OSCP-trained LoRA weights directory (output of exp_train_lora.ipynb)
    lora_input_dir    = "logs/OSCP/unet_lora",
    # Diffusion scheduler type: "LCM" only (TCD omitted for slim skeleton)
    scheduler_type    = "LCM",
    output_dir        = "outputv2/",

    # purification parameters
    num_inference_step = 5,
    # strength: fraction of forward diffusion noise added before denoising
    # 0 = no change, 1 = fully re-sampled image
    strength           = 0.2,
    guidance_scale     = 1.0,
    control_scale      = 0.8,    # ControlNet conditioning weight

    # evaluation parameters
    classifier         = "resnet50",
    num_validation_set = 10,   # number of test samples to evaluate
    seed               = 3407,
    save_images        = True,
    uconn_print_option = "postprint",

    # attack parameters
    attack_method      = "Linf_pgd",  # "Linf_pgd" | "AutoAttack"
    eps                = 8,           # pixel units /255
    attack_iter        = 10,
    alpha              = 1,           # PGD step size, pixel units /255
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device : {device}")
print(f"Attack : {config['attack_method']}  eps={config['eps']}/255")
print(f"LoRA   : {config['lora_input_dir']}")

Device : cuda
Attack : Linf_pgd  eps=8/255
LoRA   : logs/OSCP/unet_lora


## Cell 3 — Load Classifier

In [22]:
classifier = get_archs(config["classifier"], "uconn")
classifier.eval().to(device)

# Sanity check: classifier should produce [1, num_classes] logits
with torch.no_grad():
    dummy = torch.rand(1, 3, 224, 224, device=device)
    logits = classifier(dummy)
print(f"Classifier output shape: {logits.shape}  (expect [1, num_classes])")

Classifier output shape: torch.Size([1, 2])  (expect [1, num_classes])


## Cell 4 — Load SD 1.5 + ControlNet + LCM Scheduler + OSCP LoRA

Pipeline: `x` (image) + Canny edge map (`control_image`) -> `StableDiffusionControlNetImg2ImgPipeline` -> purified `x`.

LoRA loading follows `test.py`: strip the `"base_model.model."` prefix from PEFT keys before passing to `pipe.load_lora_weights()`.

In [23]:
# 1. ControlNet (canny edge conditioning)
controlnet = ControlNetModel.from_pretrained(
    config["controlnet_model"], torch_dtype=torch.float16
)

# 2. SD 1.5 img2img pipeline with ControlNet
pipe = StableDiffusionControlNetImg2ImgPipeline.from_pretrained(
    config["pretrained_model"],
    controlnet=controlnet,
    torch_dtype=torch.float16,
    safety_checker=None,
    variant="fp16",
).to(device, dtype=torch.float32)

# 3. Swap scheduler to LCM (fast 1–5 step sampling)
if config["scheduler_type"] == "LCM":
    pipe.scheduler = LCMScheduler.from_config(pipe.scheduler.config)
else:
    raise ValueError(f"Unsupported scheduler_type: {config['scheduler_type']}")

# 4. Load OSCP-trained LoRA weights
# pipe.lora_state_dict() returns (state_dict, network_alphas); we need the first element.
# PEFT wraps keys with "base_model.model." which must be stripped for direct load.
lora_state_dict, _ = pipe.lora_state_dict(config["lora_input_dir"])
unwrapped_lora = {
    key.replace("base_model.model.", ""): weight.to(pipe.dtype)
    for key, weight in lora_state_dict.items()
}
pipe.load_lora_weights(unwrapped_lora)

print("Pipeline ready.")
print(f"  Scheduler : {pipe.scheduler.__class__.__name__}")
print(f"  LoRA keys : {len(unwrapped_lora)}")

d:\MLSEC-Final\CSC592-MLSEC-InstantPure\.venv\Lib\site-packages\huggingface_hub\utils\_validators.py:205: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `hf_hub_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(
Loading weights: 100%|██████████| 196/196 [00:00<00:00, 16541.92it/s]
CLIPTextModel LOAD REPORT from: C:\Users\epuls\.cache\huggingface\hub\models--stable-diffusion-v1-5--stable-diffusion-v1-5\snapshots\451f4fe16113bff5a5d2269ed5ad43b0592e9a14\text_encoder
Key                                | Status     |  | 
-----------------------------------+------------+--+-
text_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Loading pipeline components...: 100%|██████████| 6/6 [00:00<00:00, 38.34it/s]
You have disabled the safety checker for <class 'diffusers.pipelines.controlnet.pipeline_control

Pipeline ready.
  Scheduler : LCMScheduler
  LoRA keys : 556


## Cell 5 — Helper Functions

In [ ]:
def seed_everything(seed: int = 3407):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

# Purify image(s) via ControlNet + LCM-LoRA img2img diffusion.
def lcm_lora_denoise(x, pipe, device, config):
    batch = x.shape[0]
    prompt = ["" for _ in range(batch)]
    original_size = x.shape[-1]  # assume square input

    # upsample to 512 if needed (pipeline expects 512×512)
    if original_size != 512:
        x = F.interpolate(x, size=(512, 512), mode="bilinear", align_corners=False)

    # build Canny edge map for ControlNet conditioning
    pil_x = TF.to_pil_image(x.squeeze(0))
    image_np = np.array(pil_x)
    canny = cv2.Canny(image_np, 100, 200)          # single-channel edge map
    canny_rgb = np.stack([canny, canny, canny], axis=2)  # replicate to 3ch
    control_image = Image.fromarray(canny_rgb)
    control_image = control_image.resize((512, 512), resample=Image.NEAREST)

    x = torch.clamp(x, 0.0, 1.0)
    generator = torch.Generator(device="cpu").manual_seed(config["seed"])

    # ControlNet + LCM-LoRA img2img purification
    # strength controls how much noise is injected before denoising:
    #   0 -> no change, 1 -> fully re-sampled
    pipe_out = pipe(
        prompt=prompt,
        image=x,
        control_image=control_image,
        num_inference_steps=config["num_inference_step"],
        guidance_scale=config["guidance_scale"],
        strength=config["strength"],
        controlnet_conditioning_scale=config["control_scale"],
        generator=generator,
        output_type="pt",
        return_dict=False,
    )

    # downsample output back to original resolution
    out_image = F.interpolate(
        pipe_out[0], size=(original_size, original_size), mode="bilinear", align_corners=False
    )
    return out_image, control_image.resize((original_size, original_size))


def generate_x_adv(x, y, classifier, config, device):
    # convert pixel-unit eps/alpha to float-range [0, 1]
    pgd_conf = gen_pgd_confs(
        eps=config["eps"],
        alpha=config["alpha"],
        iter=config["attack_iter"],
        input_range=(0, 1),
    )

    if config["attack_method"] == "Linf_pgd":
        adversary = LinfPGDAttack(
            classifier,
            loss_fn=torch.nn.CrossEntropyLoss(reduction="sum"),
            eps=pgd_conf["eps"],
            nb_iter=pgd_conf["iter"],
            eps_iter=pgd_conf["alpha"],
            rand_init=False,
            targeted=False,
        )
        x_adv = adversary.perturb(x, y)

    elif config["attack_method"] == "AutoAttack":
        if not _AUTOATTACK_AVAILABLE:
            raise ImportError("autoattack is not installed. Run: pip install autoattack")
        adversary = AutoAttack(
            classifier,
            norm="Linf",
            eps=pgd_conf["eps"],
            version="standard",
            device=device,
        )
        x_adv = adversary.run_standard_evaluation(x, y, bs=1)

    else:
        raise NotImplementedError(
            f"attack_method='{config['attack_method']}' not implemented. "
            "Choose 'Linf_pgd' or 'AutoAttack'."
        )

    return x_adv.to(device)


print("Helper functions defined.")

Helper functions defined.


## Cell 6 — Test Dataset + DataLoader

In [ ]:
test_dataset = get_dataset("uconn", split="test", adv=False, uconn_print_option=config["uconn_print_option"])

test_loader = DataLoader(
    test_dataset,
    batch_size=1,          # test.py always uses batch_size=1 for evaluation
    shuffle=False,
    num_workers=2,
    pin_memory=(device.type == "cuda"),
)

print(f"Test dataset size : {len(test_dataset)}")
print(f"Validating on     : {min(len(test_dataset), config['num_validation_set'])} samples")

Test dataset size : 10652
Validating on     : 10 samples


## Cell 7 — Evaluation Loop

| Metric | Description |
|--------|-------------|
| `classifier_accuracy` | Accuracy of raw classifier on **clean** images (no purification) |
| `attack_fail_rate` | Fraction of adversarial samples the raw classifier **still classifies correctly** (attack failure) |
| `clean_accuracy` | Accuracy after purifying **clean** images (measures purification distortion) |
| `robust_accuracy` | Accuracy after purifying **adversarial** images (the main OSCP robustness metric) |

In [ ]:
# output directory
lora_tag = os.path.basename(os.path.normpath(config["lora_input_dir"]))
denoise_tag = (
    f"num_inference_step_{config['num_inference_step']}"
    f"_strength_{int(config['strength'] * 1000)}"
    f"_guidance_scale_{config['guidance_scale']}"
    f"_{config['num_validation_set']}"
    f"_control_scale_{config['control_scale']}"
)
save_path = os.path.join(
    config["output_dir"],
    config["attack_method"],
    config["classifier"],
    config["scheduler_type"],
    lora_tag,
    denoise_tag,
)

if config["save_images"]:
    for sub in ("clean_image", "adversarial_image", "robust_image", "visualization"):
        os.makedirs(os.path.join(save_path, sub), exist_ok=True)

print(f"Results will be saved to:\n  {save_path}")

# counters for metrics
classifier_accuracy = 0
attack_fail_rate    = 0
clean_accuracy      = 0
robust_accuracy     = 0
processed           = 0

seed_everything(config["seed"])

# loop
pbar = tqdm(test_loader, desc="Evaluating", total=config["num_validation_set"])

for i, (x, y) in enumerate(pbar, start=1):
    if i > config["num_validation_set"]:
        break

    x, y = x.to(device), y.to(device)

    # 1. Raw classifier accuracy on clean image
    with torch.no_grad():
        classifier_accuracy += (y == classifier(x).argmax(1)).sum().item()

    # 2. Generate adversarial example (attacks raw classifier)
    x_adv = generate_x_adv(x, y, classifier, config, device)

    # 3. Attack fail rate (raw classifier on adversarial image)
    with torch.no_grad():
        attack_fail_rate += (y == classifier(x_adv).argmax(1)).sum().item()

    # 4. Purify clean image -> clean_accuracy
    with torch.no_grad():
        denoised_clean_x, natural_canny = lcm_lora_denoise(x, pipe, device, config)

    # 5. Purify adversarial image -> robust_accuracy (main OSCP metric)
    with torch.no_grad():
        robust_x, adv_canny = lcm_lora_denoise(x_adv, pipe, device, config)

    with torch.no_grad():
        clean_accuracy  += (y == classifier(denoised_clean_x.to(torch.float32)).argmax(1)).sum().item()
        robust_accuracy += (y == classifier(robust_x.to(torch.float32)).argmax(1)).sum().item()

    # save images
    if config["save_images"]:
        si(x,       save_path + f"/clean_image/{i}.png")
        si(x_adv,   save_path + f"/adversarial_image/{i}.png")
        si(robust_x, save_path + f"/robust_image/{i}.png")

    processed += 1
    pbar.set_postfix(
        cls_acc=f"{classifier_accuracy/processed:.3f}",
        robust=f"{robust_accuracy/processed:.3f}",
    )

print(f"\nDone. Processed {processed} samples.")

Results will be saved to:
  outputv2/Linf_pgd\resnet50\LCM\unet_lora\num_inference_step_5_strength_200_guidance_scale_1.0_10_control_scale_0.8


Evaluating: 100%|██████████| 10/10 [02:37<00:00, 15.74s/it, cls_acc=1.000, robust=0.900]


Done. Processed 10 samples.


## Cell 8 — Print Metrics

In [27]:
stat = {
    "classifier_accuracy" : classifier_accuracy / processed,
    "attack_fail_rate"    : attack_fail_rate    / processed,
    "clean_accuracy"      : clean_accuracy      / processed,
    "robust_accuracy"     : robust_accuracy     / processed,
}

print("\n" + "=" * 45)
print(f"  Samples evaluated : {processed}")
print(f"  Attack            : {config['attack_method']}  eps={config['eps']}/255")
print(f"  Strength          : {config['strength']}   Steps: {config['num_inference_step']}")
print("=" * 45)
for k, v in stat.items():
    print(f"  {k:<26}: {v:.4f}  ({v*100:.1f}%)")
print("=" * 45)


  Samples evaluated : 10
  Attack            : Linf_pgd  eps=8/255
  Strength          : 0.2   Steps: 5
  classifier_accuracy       : 1.0000  (100.0%)
  attack_fail_rate          : 0.0000  (0.0%)
  clean_accuracy            : 1.0000  (100.0%)
  robust_accuracy           : 0.9000  (90.0%)


## Cell 9 — Save CSV

In [ ]:
os.makedirs(save_path, exist_ok=True)
csv_path = os.path.join(save_path, "stat.csv")

df = pd.DataFrame(stat, index=[0])
df.to_csv(csv_path, index=False)

print(f"Results saved at: {csv_path}")
print(df.to_string(index=False))

Results saved → outputv2/Linf_pgd\resnet50\LCM\unet_lora\num_inference_step_5_strength_200_guidance_scale_1.0_10_control_scale_0.8\stat.csv
 classifier_accuracy  attack_fail_rate  clean_accuracy  robust_accuracy
                 1.0               0.0             1.0              0.9
